# Assignment 2 — Bag-of-Words Text Classification with a Neural Network
**Available:** Saturday, **November 21, 2026, 09:00**  **Deadline:** **December 4, 2026, 23:00** (submit via **Stud.IP Vips section**)

In this assignment, we will build a Bag-of-Words (BoW) text classifier for the Twitter Sentiment dataset. The classifier predicts whether a tweet expresses **positive sentiment (label 1)** or **negative sentiment (label 0)**.

The assignment has two connected goals:

1. Prepare text data and represent each tweet as a numerical BoW vector.
2. **Build a feed-forward neural network from scratch using NumPy**, train it with mini-batch gradient descent, and evaluate its predictions.

You will implement the network's parameter initialization, forward pass, loss, backpropagation, and parameter updates. You may use NumPy for numerical operations, pandas for reading the dataset, NLTK for text preprocessing, and scikit-learn for evaluation metrics. **Do not use PyTorch, TensorFlow, Keras, or automatic differentiation for the neural network.**

## Grading and submission

**Total: 100 points.** The assignment is published on **21 November 2026**. Submit your completed notebook by **4 December 2026, 23:00**

| Part | Points |
|---|---:|
| 1. Linear algebra and backpropagation warm-up | 15 |
| 2. Text preprocessing | 15 |
| 2.2 Vocabulary construction | 10 |
| 3. Dataset preparation and splitting | 10 |
| 4. Build and train the BoW neural network | 40 |
| 5. Model evaluation and interpretation | 10 |
| **Total** | **100** |

Submit one completed notebook. Run all cells from top to bottom before submitting. Add comments to explain your implementation and provide short written answers where requested.

# 1. Linear Algebra

## Exercise 1.1 — Neural network backpropagation warm-up (15 points)

### Task: implement a small neural network by hand

You are given a network with:
- Input layer: 2 features
- Hidden layer: 3 neurons with sigmoid activation
- Output layer: 1 neuron with sigmoid activation

Complete the forward pass, mean squared error (MSE), and backward pass. Calculate gradients for both weight matrices. This small example prepares you for implementing the larger BoW classifier in Section 4.

Use the chain rule. For sigmoid, \(\sigma(z)=1/(1+e^{-z})\) and \(\sigma'(z)=\sigma(z)(1-\sigma(z))\).

**Points:** forward pass (4), loss and loss derivative (3), backpropagation and weight gradients (6), readable comments and correct shapes (2).

Fill in the placeholders (`# YOUR CODE HERE`). Add comments explaining what each calculation represents.

The code cell is guarded: while the forward-pass placeholders are still `None`, it prints a reminder instead of interrupting **Run all**. After completing the exercise, rerun that cell to see the loss and gradient shapes.

In [1]:
import numpy as np
from numpy import integer

np.random.seed(42)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative_from_activation(a):
    # Derivative of sigmoid when its output a is already known.
    return a * (1 - a)

X = np.array([[0.5, 0.2]])
W1 = np.random.rand(2, 3)
b1 = np.zeros((1, 3))
W2 = np.random.rand(3, 1)
b2 = np.zeros((1, 1))
y_true = np.array([[1.0]])

# Forward pass
z1 = X @ W1 + b1
a1 = sigmoid(z1)
z2 = a1 @ W2 + b2 # Repeating the same pattern as the first layer, with a1 as Input, and weight & bias from layer 2
y_pred = sigmoid(z2) # Using Sigmoid to output a probability value between 0 and 1 for y
# Do not stop the whole notebook when this exercise is still unfinished.
# Complete the two forward-pass placeholders and rerun this cell to continue.
if z2 is None or y_pred is None:
    print(
        "Exercise 1.1 is not complete yet. Fill in z2 and y_pred, then rerun "
        "this cell to calculate the loss and gradients. The notebook can "
        "continue to Section 2 in the meantime."
    )
else:
    # Mean squared error
    loss = np.mean((y_true - y_pred) ** 2)

    # Backward pass: derivative of mean squared error with respect to y_pred
    d_loss = (2* (y_true - y_pred) * (-1)) / y_pred.size # Derivative of the MSE? (Check comment later)
    d_z2 = d_loss * sigmoid_derivative_from_activation(y_pred) #done
    d_W2 = a1.T @ d_z2 # done
    d_b2 = np.sum(d_z2, axis = 0, keepdims= True)
    d_a1 = d_z2 @ W2.T
    d_z1 = d_a1 * sigmoid_derivative_from_activation(a1) # done
    d_W1 = X.T @ d_z1
    d_b1 = np.sum(d_z1, axis = 0, keepdims= True)

    _gradients = [d_loss, d_z2, d_W2, d_b2, d_a1, d_z1, d_W1, d_b1]
    if any(gradient is None for gradient in _gradients):
        print(
            "The forward pass is complete. Finish all backward-pass "
            "placeholders, then rerun this cell to display the loss and "
            "gradient shapes."
        )
    else:
        print("Loss:", loss)
        print("Gradient W1 shape:", d_W1.shape)
        print("Gradient W2 shape:", d_W2.shape)


Loss: 0.07965295997703058
Gradient W1 shape: (2, 3)
Gradient W2 shape: (3, 1)


# 2. Pre-processing

Text data must be converted into a consistent representation before it can be used by a classifier. We will clean the tweets, tokenize them, normalize word forms, and remove common stopwords.

In [2]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/benmoritzcastello/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/benmoritzcastello/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/benmoritzcastello/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [3]:
import random
import re
from collections import Counter
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, HTML
from sklearn.metrics import classification_report, precision_recall_fscore_support
from tqdm.notebook import tqdm

from nltk.corpus import stopwords
from nltk.tokenize import wordpunct_tokenize
from nltk.stem import WordNetLemmatizer

%matplotlib inline
plt.style.use('ggplot')
random.seed(42)
np.random.seed(42)

## Download and upload the Twitter Sentiment dataset

**You need the `twitter_reviews.csv` file for this assignment.** Download it to your computer from Studip.

Then upload the downloaded CSV into your notebook environment:

- **Google Colab:** run the next cell and select the downloaded CSV when the file picker opens.
- **Jupyter Notebook:** place `twitter_reviews.csv` in the same folder as this notebook.

The CSV must contain the columns `tweet` (text) and `label` (0 = negative, 1 = positive).


In [4]:
from pathlib import Path
import pandas as pd

DATA_PATH = "twitter_reviews.csv"

try:
    # In Google Colab, open the upload dialog so you can select the CSV you downloaded.
    from google.colab import files

    print("Please select the downloaded twitter_reviews.csv file.")
    uploaded_files = files.upload()

    csv_files = [
        filename for filename in uploaded_files
        if filename.lower().endswith(".csv")
    ]
    if not csv_files:
        raise ValueError(
            "No CSV file was uploaded. Please download twitter_reviews.csv "
            "from the Google Drive link above and upload it here."
        )

    DATA_PATH = csv_files[0]

except ImportError:
    # In local Jupyter, put twitter_reviews.csv next to this notebook.
    if not Path(DATA_PATH).is_file():
        raise FileNotFoundError(
            "twitter_reviews.csv was not found. Download it from the Google Drive "
            "link above and place it in the same folder as this notebook."
        )

# Verify the expected columns before continuing.
_preview = pd.read_csv(DATA_PATH, nrows=5)
_required_columns = {"tweet", "label"}
_missing_columns = _required_columns.difference(_preview.columns)

if _missing_columns:
    raise ValueError(
        f"The uploaded CSV is missing required columns: {sorted(_missing_columns)}. "
        "Please upload the correct twitter_reviews.csv file."
    )

print(f"Dataset uploaded successfully: {DATA_PATH}")
print("Dataset columns:", list(_preview.columns))


Dataset uploaded successfully: twitter_reviews.csv
Dataset columns: ['Unnamed: 0', 'tweet', 'label']


### Take a look at a few examples

The labels are binary: **1 = positive** and **0 = negative**. Inspect a few rows and check whether the labels and tweet text look as expected.

In [5]:
df = pd.read_csv(DATA_PATH)
df.sample(min(5, len(df)), random_state=42)

,Unnamed: 0,tweet,label
987231,987231,Just asked my friend if we were in a non-smoki...,0
79954,79954,@Cantib ajarin ngetweet donk,0
567130,567130,Already.I'll block all the people who says: YO...,1
500891,500891,"@Sue_15 monsoon has arrived in Goa, but its st...",1
55399,55399,@basedmagazine My phone would go dead right af...,0


In [6]:
print('Number of records:', len(df))
print('Number of positive tweets:', int((df.label == 1).sum()))
print('Number of negative tweets:', int((df.label == 0).sum()))
print('\nExample negative tweet:')
print(df.loc[df.label == 0, 'tweet'].iloc[0])
print('\nExample positive tweet:')
print(df.loc[df.label == 1, 'tweet'].iloc[0])

Number of records: 1000000
Number of positive tweets: 531354
Number of negative tweets: 468646

Example negative tweet:
                     is so sad for my APL friend.............

Example positive tweet:
              omg its already 7:30 :O


## 2.1 Preprocess text

We will apply the following operations:
- Replace or remove special characters.
- Convert text to lowercase.
- Tokenize text into words.
- Lemmatize tokens (nouns first, then verbs).
- Replace standalone numbers with a special token.
- Remove stopwords.

The order matters: each operation prepares the text for the next one.

The following short example is used to test the preprocessing functions before applying them to the full dataset.

In [7]:
test_corpus = """Curtain drops at the #Crown Uptown after 32 years
http://www.kansas.com/news/featured/story/860875.html
The actors were running and the performances were amazing!"""
print(test_corpus)

Curtain drops at the #Crown Uptown after 32 years
http://www.kansas.com/news/featured/story/860875.html
The actors were running and the performances were amazing!


### Exercise 2.1.1 — Remove special characters and lowercase (3 points)

Write `special_and_lower(text)`. It should lowercase the text and remove or replace punctuation/special characters while preserving letters, digits, and whitespace. Avoid concatenating words that were separated by punctuation.

In [8]:
def special_and_lower(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

test_corpus = special_and_lower(test_corpus)
print(test_corpus)

curtain drops at the crown uptown after 32 years http www kansas com news featured story 860875 html the actors were running and the performances were amazing


### Exercise 2.1.2 — Tokenization (3 points)

Write `tokenize_text(text)` using NLTK's `wordpunct_tokenize`. It should return a list of tokens.

In [9]:
def tokenize_text(text):
    text = nltk.wordpunct_tokenize(text)
    return text

test_corpus = tokenize_text(test_corpus)
print(test_corpus)

['curtain', 'drops', 'at', 'the', 'crown', 'uptown', 'after', '32', 'years', 'http', 'www', 'kansas', 'com', 'news', 'featured', 'story', '860875', 'html', 'the', 'actors', 'were', 'running', 'and', 'the', 'performances', 'were', 'amazing']


### Exercise 2.1.3 — Lemmatization (3 points)

Using `WordNetLemmatizer`, write `lemmatize_text(tokens)`. Lemmatize each token as a noun first and then as a verb. For example, plural nouns and inflected verbs should be mapped to their base forms where WordNet supports the transformation.

In [10]:

lemmatizer = WordNetLemmatizer()

def lemmatize_text(tokens):
    result = []
    for token in tokens:
        lem = lemmatizer.lemmatize(token, pos="n")
        lem = lemmatizer.lemmatize(lem, pos="v")
        result.append(lem)
    tokens = result
    return tokens

test_corpus = lemmatize_text(test_corpus)
print(test_corpus)

['curtain', 'drop', 'at', 'the', 'crown', 'uptown', 'after', '32', 'year', 'http', 'www', 'kansa', 'com', 'news', 'feature', 'story', '860875', 'html', 'the', 'actor', 'be', 'run', 'and', 'the', 'performance', 'be', 'amaze']


### Exercise 2.1.4 — Replace standalone numbers (3 points)

Write `replace_numbers(tokens)` to replace any token consisting entirely of digits with the special token `'<NUM>'`. A token such as `'42'` should be replaced, while `'42h'` should remain unchanged.

In [11]:
def replace_numbers(tokens):
    result = []
    for token in tokens:
        if token.isdigit():
            result.append("<NUM>")
        else:
            result.append(token)
        tokens = result
        return tokens

test_corpus = replace_numbers(test_corpus)
print(test_corpus)


['curtain']


In [12]:
def replace_numbers(tokens):
    # YOUR CODE HERE
    return tokens

test_corpus = replace_numbers(test_corpus)
print(test_corpus)

['curtain']


### Exercise 2.1.5 — Remove stopwords (3 points)

Write `remove_stopwords(tokens)` using the NLTK English stopword list. Return a list containing only tokens that are not stopwords. Do not remove the special token `'<NUM>'`.

In [13]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [token for token in tokens if token not in stop_words]
    return tokens

test_corpus = remove_stopwords(test_corpus)
print(test_corpus)

['curtain']


# 2.2 Building the Vocabulary

A vocabulary maps each token to a unique integer index. This mapping is necessary because a neural network operates on numerical inputs rather than strings.

### Exercise 2.2.1 — Build token-to-index vocabulary (5 points)

Define `build_vocab(corpus)`. The input is a list of tokenized documents (a list of lists). Return a dictionary in which each unique token is a key and its integer index is the value. Each token must receive exactly one index.

In [14]:
def build_vocab(corpus):
    vocab = {}
    for doc in corpus:
        for token in doc:
            if token not in vocab:
                vocab[token] = len(vocab)
    return vocab

print(build_vocab([test_corpus]))

{'curtain': 0}


### Exercise 2.2.2 — Build index-to-token vocabulary (5 points)

Define `build_index2token(vocab)`. Return the inverse mapping: integer indexes as keys and tokens as values. The indexes must remain exactly the same as in `vocab`.

In [15]:
def build_index2token(vocab):
    index2token = {}
    for token, idx in vocab.items():
        index2token[idx] = token
    return index2token

print(build_index2token(build_vocab([test_corpus])))

{0: 'curtain'}


## Package the preprocessing steps

The next cell combines the individual functions into one pipeline. The BoW vector counts how often each vocabulary item occurs in a document.

In [16]:
def remove_rare_words(tokens, common_tokens, max_len):
    return [token if token in common_tokens else '<UNK>' for token in tokens][-max_len:]

def tokenize(text, stop_words, lemmatizer):
    text = special_and_lower(text)
    tokens = tokenize_text(text)
    tokens = lemmatize_text(tokens)
    tokens = replace_numbers(tokens)
    tokens = remove_stopwords(tokens)
    return tokens

def build_bow_vector(sequence, idx2token):
    vector = [0] * len(idx2token)
    for token_idx in sequence:
        if token_idx not in idx2token:
            raise ValueError('Wrong sequence index found!')
        vector[token_idx] += 1
    return vector

# 3. Splitting and transforming the Dataset

### Exercise 3.1 — Explore vocabulary and sequence parameters (not scored)

Experiment with `MAX_LEN` and `MAX_VOCAB`. `MAX_LEN` limits the number of tokens retained per tweet; `MAX_VOCAB` limits the vocabulary to the most frequent tokens. Record at least two configurations and compare their validation performance after training. Consider how vocabulary size affects the dimensionality of the network's input layer.

In [17]:
MAX_LEN = 128
MAX_VOCAB = 1000

We now package the processed examples into a dataset-like Python object. Each item contains:
1. the indexed token sequence;
2. its Bag-of-Words vector;
3. its sentiment label;
4. the original tweet text.

The implementation below uses ordinary Python and NumPy data structures; no deep-learning framework is needed.

In [18]:
class TwitterDataset:
    def __init__(self, data_path, max_vocab=1000, max_len=128):
        frame = pd.read_csv(data_path)
        stop_words_local = set(stopwords.words('english'))
        lemmatizer_local = WordNetLemmatizer()

        frame['tokens'] = frame.tweet.fillna('').apply(
            lambda text: tokenize(text, stop_words_local, lemmatizer_local)
        )
        all_tokens = [token for doc in frame.tokens for token in doc]
        common_tokens = set(token for token, _ in Counter(all_tokens).most_common(max_vocab))

        frame['tokens'] = frame.tokens.apply(
            lambda tokens: remove_rare_words(tokens, common_tokens, max_len)
        )
        frame = frame[frame.tokens.apply(lambda tokens: len(tokens) > 0)].copy()

        self.token2idx = build_vocab(frame.tokens.tolist())
        self.idx2token = build_index2token(self.token2idx)
        frame['indexed_tokens'] = frame.tokens.apply(
            lambda doc: [self.token2idx[token] for token in doc]
        )
        frame['bow_vector'] = frame.indexed_tokens.apply(
            lambda seq: build_bow_vector(seq, self.idx2token)
        )

        self.text = frame.tweet.tolist()
        self.sequences = frame.indexed_tokens.tolist()
        self.bow_vector = frame.bow_vector.tolist()
        self.targets = frame.label.astype(int).tolist()

    def __getitem__(self, index):
        return (
            self.sequences[index],
            np.asarray(self.bow_vector[index], dtype=float),
            self.targets[index],
            self.text[index],
        )

    def __len__(self):
        return len(self.targets)

### Create the processed dataset

The dataset class below uses the preprocessing functions you implemented in Section 2.


In [19]:
# Check that the functions from Section 2 have been defined before processing the CSV.
_required_functions = [
    "special_and_lower",
    "tokenize_text",
    "lemmatize_text",
    "replace_numbers",
    "remove_stopwords",
    "build_vocab",
    "build_index2token",
    "tokenize",
    "build_bow_vector",
]
_missing_functions = [name for name in _required_functions if name not in globals()]

if _missing_functions:
    raise RuntimeError(
        "Some preprocessing functions have not been defined yet: "
        + ", ".join(_missing_functions)
        + ". Run the preprocessing and vocabulary cells in Section 2 before this cell."
    )

if "DATA_PATH" not in globals():
    raise RuntimeError(
        "The CSV has not been uploaded yet. Run the 'Download and upload the "
        "Twitter Sentiment dataset' cell first."
    )

print("Preprocessing tweets; this step may take a few minutes.")
dataset = TwitterDataset(DATA_PATH, max_vocab=MAX_VOCAB, max_len=MAX_LEN)
print("Number of processed records:", len(dataset))
print("Vocabulary size:", len(dataset.token2idx))


Preprocessing tweets; this step may take a few minutes.
Number of processed records: 999843
Vocabulary size: 1001


In [20]:
# The TwitterDataset class and dataset were created together in the cell above.
if "dataset" not in globals():
    raise RuntimeError(
        "The dataset has not been created yet. Run the preceding cell that defines "
        "TwitterDataset and preprocesses the CSV, then run this cell again."
    )

print("Dataset is ready.")
print("Number of processed records:", len(dataset))
print("Vocabulary size:", len(dataset.token2idx))

Dataset is ready.
Number of processed records: 999843
Vocabulary size: 1001


### Inspect a randomly selected processed example

In [21]:
random_idx = random.randint(0, len(dataset) - 1)
sample_seq, bow_vector, sample_target, sample_text = dataset[random_idx]
print("Original tweet:", sample_text)
print("Indexed sequence (first 20):", sample_seq[:20])
print("BoW vector shape:", bow_vector.shape)
print("Sentiment label:", sample_target)

Original tweet: forgetting the time... need to go... hurry up... Damm why do al the sunday's go so fast 
Indexed sequence (first 20): [487, 142, 131, 58, 1, 1, 1, 186, 58, 739]
BoW vector shape: (1001,)
Sentiment label: 0


### Exercise 3.2 — Split into training, validation, and test sets (5 points)

Split the dataset into three non-overlapping subsets:
- **Training (80%)**: used to learn the network parameters.
- **Validation (10%)**: used to compare configurations and monitor overfitting.
- **Test (10%)**: used only for the final evaluation.

Complete `split_train_valid_test`. Ensure that the three lengths add up to the full dataset size, including when the dataset size is not divisible by ten.

In [26]:
def split_train_valid_test(corpus, valid_ratio=0.1, test_ratio=0.1, seed=42):
    n = len(corpus)
    valid_length = int(n*valid_ratio) #Muliply corpus length with the 0.1
    test_length = int(n*test_ratio) # The same
    train_length = int(n - valid_length - test_length) # Ensure the whole corpus is used by subtracting validation and test from the total amount

    indices = np.random.default_rng(seed).permutation(n)
    train_end = train_length
    valid_end = train_length + valid_length
    train_idx = indices[:train_end]
    valid_idx = indices[train_end:valid_end]
    test_idx = indices[valid_end:]
    return train_idx, valid_idx, test_idx

train_idx, valid_idx, test_idx = split_train_valid_test(dataset)
print(len(train_idx), len(valid_idx), len(test_idx))

799875 99984 99984


### Create the data splits

The next cell creates subsets using the indices returned above. Keep the test set separate until the final evaluation.

In [27]:
class IndexedExamples:
    """A lightweight view over selected dataset rows; stores indices only."""
    def __init__(self, dataset, indices):
        self.dataset = dataset
        self.indices = np.asarray(indices, dtype=int)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        return self.dataset[int(self.indices[index])]

def get_examples(dataset, indices):
    return IndexedExamples(dataset, indices)

train_examples = get_examples(dataset, train_idx)
valid_examples = get_examples(dataset, valid_idx)
test_examples = get_examples(dataset, test_idx)

### Exercise 3.3 — Mini-batch preparation (5 points)

A batch is a small group of examples processed together. Mini-batch training makes the gradient estimate less noisy than using one example at a time, while requiring less memory than processing the entire dataset at once.

Complete `make_batches`. It should shuffle the training examples when requested and return batches containing:
- a two-dimensional NumPy array of BoW vectors;
- a one-dimensional NumPy array of integer labels;
- the original text strings.

**Points:** correct shuffling when requested (1), correct batch boundaries and final partial batch (1), correctly shaped BoW matrix (1), correctly aligned labels (1), retaining corresponding tweet texts (1).

In [31]:
BATCH_SIZE = 128

def make_batches(examples, batch_size=128, shuffle=False):
    indices = np.arange(len(examples))
    if shuffle:
        np.random.shuffle(indices)

    batches = []
    for start in range(0, len(examples), batch_size):
        batch_idx = indices[start : start + batch_size]

        bows, labels, texts = [], [], []
        for i in batch_idx:
            seq, bow, label, text = examples[i]
            bows.append(bow)
            labels.append(label)
            texts.append(text)

        X_batch = np.array(bows)
        y_batch = np.array(labels, dtype = int)
        batches.append((X_batch, y_batch, texts))
    return batches

train_batches = make_batches(train_examples, BATCH_SIZE, shuffle=True)
valid_batches = make_batches(valid_examples, BATCH_SIZE, shuffle=False)
test_batches = make_batches(test_examples, BATCH_SIZE, shuffle=False)
print("Number of training batches:", len(train_batches))

Number of training batches: 6250


### Inspect one batch

In [32]:
if train_batches:
    X_batch, y_batch, text_batch = train_batches[0]
    print("BoW batch shape:", X_batch.shape)
    print("Label batch shape:", y_batch.shape)
    print("Example text:", text_batch[0])
    print("Example label:", y_batch[0])
else:
    print("Complete the make_batches exercise above, then rerun this cell.")

BoW batch shape: (128, 1001)
Label batch shape: (128,)
Example text: Heading home today! Goodbye Florida! 
Example label: 0


# 4. BoW Model — Build the neural network from scratch

## Build a feed-forward BoW classifier

Each tweet is represented by a BoW vector of length `vocab_size`. The network has two hidden layers and an output layer with two scores: one for negative sentiment and one for positive sentiment.

You will implement this network yourself with NumPy. You must write the forward pass, softmax, cross-entropy loss, backpropagation, and gradient-descent updates. The notebook does not provide a pre-built neural-network class, and you must not use automatic differentiation.

For a two-hidden-layer network:

\[
Z_1=XW_1+b_1,\quad A_1=f(Z_1)
\]
\[
Z_2=A_1W_2+b_2,\quad A_2=f(Z_2)
\]
\[
Z_3=A_2W_3+b_3,\quad P=softmax(Z_3)
\]

The output is a two-dimensional probability distribution for each tweet.

### Exercise 4.1 — Choose and implement an activation function (5 points)

Implement at least one non-linear activation function and its derivative. You may choose ReLU, sigmoid, or tanh. Use the same activation function in both hidden layers. Explain briefly why a non-linear activation is needed.

**Hint:** For ReLU, \(f(z)=max(0,z)\), and its derivative is 1 for positive inputs and 0 otherwise.

In [37]:
def activation(z, name="relu"):
    return np.maximum(0, z)

def activation_derivative(z, name="relu"):
    return (z > 0).astype(float)

# You may change this after experimenting.
ACTIVATION = "relu"

### Exercise 4.2 — Experiment with hidden-layer sizes (not scored)

Experiment with different values for `HIDDEN1` and `HIDDEN2`, for example 32/16, 64/32, and 128/64. Larger layers have more parameters and may take longer to train or overfit. Use validation loss and validation accuracy to compare configurations. Do not use the test set to select hyperparameters.

In [38]:
HIDDEN1 = 64
HIDDEN2 = 32

### Exercise 4.3 — Build the two-hidden-layer network from scratch (10 points)

Complete `initialize_parameters`. Initialize three weight matrices and three bias vectors with shapes consistent with the architecture. Use small random weights and zero biases.

Then implement the forward pass in `forward`. It must return the class probabilities and a cache containing intermediate values needed for backpropagation.

The final model has **two hidden layers**. You implement both layers and their forward computation yourself. Change the hidden-layer sizes and verify that the code still works.

In [45]:
def initialize_parameters(input_size, hidden1, hidden2, output_size, seed=42):
    rng = np.random.default_rng(seed)
    params = {}
    params["W1"] = rng.normal(0, 0.01, size = (input_size, hidden1))
    params["b1"] = np.zeros((1, hidden1))
    params["W2"] = rng.normal(0, 0.01, size = (hidden1, hidden2))
    params["b2"] = np.zeros((1, hidden2))
    params["W3"] = rng.normal(0, 0.01, size = (hidden2, output_size))
    params["b3"] = np.zeros((1, output_size))
    return params

def softmax(logits):
    shifted = logits - np.max(logits, axis = 1, keepdims=True)
    exp_scores = np.exp(shifted)
    return exp_scores / np.sum(exp_scores, axis = 1, keepdims = True)

def forward(X, params, activation_name="relu"):
    Z1 = X @ params["W1"] + params["b1"]
    A1 = activation(Z1, activation_name)
    Z2 = A1 @ params["W2"] + params["b2"]
    A2 = activation(Z2, activation_name)
    Z3 = A2 @ params["W3"] + params["b3"]
    probabilites = softmax(Z3)

    cache = {"X": X, "Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}
    return probabilites, cache

params = initialize_parameters(
    input_size=len(dataset.token2idx),
    hidden1=HIDDEN1,
    hidden2=HIDDEN2,
    output_size=2,
)

### Exercise 4.4 — Loss, backpropagation, and parameter updates (15 points)

Implement the multi-class cross-entropy loss and the backward pass. For a batch of `m` examples, the softmax-cross-entropy gradient with respect to the output logits is:

\[
dZ_3 = \frac{P-Y_{onehot}}{m}
\]

Use the chain rule to propagate this gradient through the output layer and both hidden layers. Return a gradient for every parameter.

**Points:** loss function (3), output-layer gradients (3), hidden-layer gradients (6), correct gradient shapes and stable numerical computation (3).

In [49]:
def cross_entropy_loss(probabilities, y):
    m = len(y)
    correct_probs = probabilities[np.arange(m), y]
    return -np.mean(np.log(correct_probs + 1e-12))

def backward(X, y, probabilities, cache, params, activation_name="relu"):
    m = len(y)
    A1, A2 = cache ["A1"], cache["A2"]
    Z1, Z2 = cache ["Z1"], cache ["Z2"]

    Y_onehot = np.zeros_like(probabilities)
    Y_onehot[np.arange(m), y] = 1
    dZ3 = (probabilities - Y_onehot) / m

    dW3 = A2.T @dZ3
    db3 = np.sum(dZ3, axis = 0, keepdims= True)
    dA2 = dZ3 @params["W3"].T

    dZ2 = dA2 * activation_derivative(Z2, activation_name)
    dW2 = A1.T @ dZ2
    db2 = np.sum(dZ2, axis = 0, keepdims=True)
    dA1 = dZ2 @params["W2"].T

    dZ1 = dA1 * activation_derivative(Z1, activation_name)
    dW1 = X.T @ dZ1
    db1 = np.sum(dZ1, axis = 0, keepdims= True)

    return {"W1": dW1, "b1": db1, "W2": dW2, "b2":db2, "W3": dW3, "b3": db3}

### Exercise 4.5 — Train using mini-batch gradient descent (10 points)

Implement `update_parameters` and the training loop. For every mini-batch:
1. Run the forward pass.
2. Calculate cross-entropy loss.
3. Calculate gradients with backpropagation.
4. Update every weight and bias using gradient descent:

\[
\theta \leftarrow \theta - \alpha\nabla_\theta L
\]

Use the validation set to monitor performance. The test set must not be used during training.

In [50]:
LEARNING_RATE = 0.05
MAX_EPOCHS = 20

def update_parameters(params, grads, learning_rate):
    for key in params:
        params[key] = params[key] - learning_rate * grads[key]
    return params

def evaluate_loss(examples, params, batch_size=BATCH_SIZE):
    batches = make_batches(examples, batch_size, shuffle=False)
    total_loss = 0.0
    total_examples = 0
    for X_batch, y_batch, _ in batches:
        probs, _ = forward(X_batch, params, ACTIVATION)
        total_loss += cross_entropy_loss(probs, y_batch) * len(y_batch)
        total_examples += len(y_batch)
    return total_loss / max(total_examples, 1)

def train_network(train_examples, valid_examples, params,
                  learning_rate=LEARNING_RATE, max_epochs=MAX_EPOCHS):
    train_losses, valid_losses = [], []
    for epoch in range(max_epochs):
        epoch_loss = 0.0
        seen = 0
        for X_batch, y_batch, _ in make_batches(train_examples, BATCH_SIZE, shuffle=True):
            probs, cache = forward(X_batch, params, ACTIVATION)
            loss = cross_entropy_loss(probs, y_batch)
            grads = backward(X_batch, y_batch, probs, cache, params, ACTIVATION)
            params = update_parameters(params, grads, learning_rate)
            epoch_loss += loss * len(y_batch)
            seen += len(y_batch)

        train_loss = epoch_loss / max(seen, 1)
        valid_loss = evaluate_loss(valid_examples, params)
        train_losses.append(train_loss)
        valid_losses.append(valid_loss)
        print(f"Epoch {epoch+1:02d}/{max_epochs} | train loss={train_loss:.4f} | valid loss={valid_loss:.4f}")

    return params, train_losses, valid_losses

# Run training after completing the functions above.
# params, train_losses, valid_losses = train_network(train_examples, valid_examples, params)

### Exercise 4.6 — Learning-rate investigation (not scored)

What happens if you set a very small learning rate? What if you set a very large learning rate? Try at least two values and compare the loss curves. Explain your observations in 2–4 sentences.

In [ ]:
# Example: after completing training, plot the recorded losses.
# plt.plot(train_losses, label="training")
# plt.plot(valid_losses, label="validation")
# plt.xlabel("Epoch")
# plt.ylabel("Cross-entropy loss")
# plt.legend()
# plt.show()

### Training notes: loss and gradient descent

The criterion measures the difference between predicted probabilities and the true class. Gradient descent updates the model parameters in the direction that reduces the loss. In this assignment, you implement both the gradient calculation and the update directly, rather than calling an optimizer library.

A decreasing training loss is generally expected. If training loss decreases while validation loss increases, the network may be overfitting.

### Optional extension — learning-rate schedule (not scored)

You may implement a simple learning-rate decay, such as multiplying the learning rate by 0.95 after each epoch. This is optional and is not required for full credit.

In [ ]:
# Optional extension:
# current_lr = LEARNING_RATE
# current_lr *= 0.95

# 5. Check Performance of BoW Model

The final evaluation uses the held-out test set. Do not use test results to choose the architecture or learning rate. For each example, the predicted class is the index with the highest output probability.

### Exercise 5.1 — Precision, recall, and F1-score (10 points)

Calculate precision, recall, and F1-score for both classes (negative and positive). Also report test accuracy. You may use `sklearn.metrics` for evaluation; the neural network itself must remain a NumPy implementation.

**Points:** correct test predictions (2), precision and recall for both classes (4), F1-scores (2), accuracy and interpretation (2).

In [51]:
def predict(examples, params):
    all_predictions = []
    all_probabilities = []
    for X_batch, y_batch, _ in make_batches(examples, BATCH_SIZE, shuffle=False):
        probabilities, _ = forward(X_batch, params, ACTIVATION)
        all_probabilities.extend(probabilities)
        all_predictions.extend(np.argmax(probabilities, axis=1))
    return np.asarray(all_predictions), np.asarray(all_probabilities)

# Run after training:
# y_pred, y_prob = predict(test_examples, params)
# y_true = np.asarray([example[2] for example in test_examples])
# print(classification_report(y_true, y_pred, target_names=["negative", "positive"], zero_division=0))

### Inspect a few predictions

Compare the original tweet, predicted class, and true label. Consider examples where the model is incorrect. Does the tweet contain negation, sarcasm, or words that may be misleading when represented as an unordered BoW vector?

### Short reflection (included in the Section 5 points)

Answer briefly:
1. What information is lost when a tweet is represented as a Bag-of-Words vector?
2. Why do we need a validation set in addition to a training set and a test set?
3. Give one example of a sentiment expression that may be difficult for a BoW classifier.
